In [6]:
import sys
sys.path.insert(0, r"C:\CYME\CYME")
import cympy

from datetime import datetime
import csv
import os
import math
import copy
import re
import decimal
import logging

def run_cyme(sxrt):

    
    print("Opening: ", sxrt)
    cympy.study.Open(sxrt)

    # ---------------------------------------------------------------------------
    # Load Flow — all active controls disabled
    # ---------------------------------------------------------------------------
    lf = cympy.sim.LoadFlow()

    # --- General solver settings ------------------------------------------------
    lf.SetValue(0.001,  'ParametersConfigurations[0].VoltageTolerance')
    lf.SetValue("VoltageDropUnbalanced", 'ParametersConfigurations[0].AnalysisMode')

    # Source / line modelling
    lf.SetValue(0, 'ParametersConfigurations[0].IncludeSourceImpedance')
    lf.SetValue(0, 'ParametersConfigurations[0].IncludeLineCharging')
    lf.SetValue(0, 'ParametersConfigurations[0].AssumeLineTransposition')

    # Temperature adjustment off
    lf.SetValue(0, 'ParametersConfigurations[0].TemperatureAdjustment.EnableTemperatureAdjustment')

    # Voltage-sensitivity load model
    lf.SetValue('Global', 'ParametersConfigurations[0].LoadFlowVoltageSensitivityLoadModel.Mode')
    lf.SetValue(0.0,     'ParametersConfigurations[0].LoadFlowVoltageSensitivityLoadModel.V')

    # --- Lock regulators / switched shunts (prevents tap & stage changes) -------
    lf.SetValue(1, 'ParametersConfigurations[0].LockMultiStageSwitchableShunts')
    lf.SetValue(1, 'ParametersConfigurations[0].LockSwitchedCapacitors')

    # --- Equipment status: disable ALL active controls --------------------------
    # Switched capacitor control modes — all OFF
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.VoltageCapStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.CurrentCapStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.ReactiveCurrentCapStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.PowerFactorCapStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.TemperatureCapStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.TimeCapStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.KVARCapStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.PythonScriptCapStatus')

    # Fixed caps remain on (they have no switching logic)
    lf.SetValue(1, 'ParametersConfigurations[0].EquipmentStatusParameters.FixedCapStatus')

    # Multi-stage switchable shunts — OFF
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.MultiStageSwitchableShuntStatus')

    # SVCs / VAR compensators — OFF
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.SVCStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.VARCompensatorStatus')

    # Motors — OFF
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.SynchronousMotorStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.InductionMotorStatus')

    # Generators — keep synchronous & ECG energised but all others OFF
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.GeneratorStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.SynchronousGeneratorStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.InductionGeneratorStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.ElectronicallyCoupledGeneratorStatus')

    # Renewables / storage — OFF
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.WecsStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.BESSStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.SofcStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.PhotovoltaicStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.MicroTurbineStatus')

    # --- Run load flow -----------------------------------------------------------
    lf.Run()
    print("Load flow complete (all active controls disabled)")

In [2]:
import pandas as pd

df = pd.read_csv(rf"c:\Users\fgonzales\git\mc-0.0.1.15\scripts\encoded - Copy.csv")

In [12]:
import numpy as np
import pandas as pd

def _safe_float(val):
    """Convert a CYME query result to float, returning None for empty/invalid strings."""
    if val is None or (isinstance(val, str) and val.strip() == ''):
        return None
    return float(val)

def _get_cyme_v_pu(sxst_file):
    """Run CYME load flow and return a DataFrame with NODE_ID, VA, VB, VC (pu)."""
    import sys
    sys.path.insert(0, r"C:\CYME\CYME")
    import cympy

    PRECISION = 7
    nodes = cympy.study.ListNodes(cympy.enums.NodeType.All)
    rows = []
    for node in nodes:
        nid = node.ID
        rows.append({
            'NODE_ID': nid,
            'VA_CYME': _safe_float(cympy.study.QueryInfoNode('VpuA', nid, PRECISION)),
            'VB_CYME': _safe_float(cympy.study.QueryInfoNode('VpuB', nid, PRECISION)),
            'VC_CYME': _safe_float(cympy.study.QueryInfoNode('VpuC', nid, PRECISION)),
        })
    return pd.DataFrame(rows)


def _get_mc_v_pu(net):
    """Extract per-bus VA, VB, VC (pu) from a solved MC net. Returns a DataFrame."""
    bus_indices = net.bus.index.get_level_values(0).unique()
    rows = []
    for bus_idx in bus_indices:
        bus_data = net.bus.loc[bus_idx]
        name = bus_data['name'].iloc[0] if isinstance(bus_data, pd.DataFrame) else bus_data['name']
        v = {}
        for phase, label in [(1, 'VA_MC'), (2, 'VB_MC'), (3, 'VC_MC')]:
            if (bus_idx, phase) in net.res_bus.index:
                vm = float(net.res_bus.loc[(bus_idx, phase), 'vm_pu'])
                v[label] = vm if not np.isnan(vm) else None
            else:
                v[label] = None
        rows.append({'NODE_ID': name, **v})
    return pd.DataFrame(rows)


def compare_mc_cyme(net, sxst_file):
    """Run both engines, compute network-wide mean v_pu independently (no node matching)."""
    from multiconductor.pycci.cci_powerflow import run_pf

    # Query CYME results (study already open)
    df_cyme = _get_cyme_v_pu(sxst_file)
    for c in ['VA_CYME', 'VB_CYME', 'VC_CYME']:
        df_cyme[c] = pd.to_numeric(df_cyme[c], errors='coerce')
    df_cyme = df_cyme.dropna(subset=['VA_CYME', 'VB_CYME', 'VC_CYME'])

    # Run MC
    run_pf(net, tol_vang_rad=1e-5, tol_vmag_pu=1e-5)
    df_mc = _get_mc_v_pu(net)
    for c in ['VA_MC', 'VB_MC', 'VC_MC']:
        df_mc[c] = pd.to_numeric(df_mc[c], errors='coerce')
    df_mc = df_mc.dropna(subset=['VA_MC', 'VB_MC', 'VC_MC'])

    summary = {
        'VA_MC_mean':   df_mc['VA_MC'].mean(),
        'VB_MC_mean':   df_mc['VB_MC'].mean(),
        'VC_MC_mean':   df_mc['VC_MC'].mean(),
        'MC_node_count': len(df_mc),
        'VA_CYME_mean': df_cyme['VA_CYME'].mean(),
        'VB_CYME_mean': df_cyme['VB_CYME'].mean(),
        'VC_CYME_mean': df_cyme['VC_CYME'].mean(),
        'CYME_node_count': len(df_cyme),
    }
    return summary

In [13]:
def run_circuit(circuit_key):
    import pickle
    from pathlib import Path

    print(f"Processing {circuit_key}...")
    pkl_file = Path(rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\{circuit_key}.pkl")
    sxst_file = Path(rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\cyme\{circuit_key}.sxst")

    # Run CYME load flow first (opens study, keeps it open for querying)
    run_cyme(str(sxst_file))

    with open(pkl_file, 'rb') as fh:
        net = pickle.load(fh)

    summary = compare_mc_cyme(net, str(sxst_file))

    # Close the CYME study after results have been queried
    cympy.study.Close(AskForSave=False)

    summary['CKT_KEY'] = circuit_key
    print(f"  {circuit_key}: MC nodes={summary['MC_node_count']} CYME nodes={summary['CYME_node_count']}, "
          f"VA mean MC={summary['VA_MC_mean']:.5f} CYME={summary['VA_CYME_mean']:.5f}")
    return summary

In [14]:
all_summaries = []
for circuit_key in df['CIRCUIT_KEY']:
    try:
        summary = run_circuit(circuit_key)
        all_summaries.append(summary)
    except Exception as e:
        print(f"Error running circuit {circuit_key}: {e}")

summary_df = pd.DataFrame(all_summaries)
summary_df

Processing CKT_2155_12331...
Opening:  c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\cyme\CKT_2155_12331.sxst
Load flow complete (all active controls disabled)
  CKT_2155_12331: MC nodes=719 CYME nodes=619, VA mean MC=0.94993 CYME=1.00550
Processing CKT_3250_02485...
Opening:  c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\cyme\CKT_3250_02485.sxst
Load flow complete (all active controls disabled)
  CKT_3250_02485: MC nodes=1399 CYME nodes=998, VA mean MC=1.01167 CYME=1.01173
Processing CKT_156_06131...
Opening:  c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\cyme\CKT_156_06131.sxst
Load flow complete (all active controls disabled)
  CKT_156_06131: MC nodes=479 CYME nodes=473, VA mean MC=0.98987 CYME=1.00306
Processing CKT_209_14375...
Opening:  c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\cyme\CKT_209_14375.sxst
Load flow complete (all active controls disabled)
  CKT_209_14375: MC nodes=829 CYME nodes=782, VA mean MC=0.96746 CYME=0.96380
Processing CKT_1727_14043...
Open

,VA_MC_mean,VB_MC_mean,VC_MC_mean,MC_node_count,VA_CYME_mean,VB_CYME_mean,VC_CYME_mean,CYME_node_count,CKT_KEY
0,0.949925,0.949001,0.990828,719,1.005505,1.005505,1.005505,619,CKT_2155_12331
1,1.011674,1.090606,1.010080,1399,1.011734,1.011734,1.011734,998,CKT_3250_02485
2,0.989875,0.977278,0.989813,479,1.003064,1.003064,1.003064,473,CKT_156_06131
3,0.967460,0.965021,0.972518,829,0.963803,0.963803,0.963803,782,CKT_209_14375
4,1.090048,1.072761,1.292763,763,1.004819,1.004819,1.004819,699,CKT_1727_14043
...,...,...,...,...,...,...,...,...,...
833,0.999293,0.999293,0.999292,391,1.010518,1.010518,1.010518,391,CKT_2082_11931
834,0.980956,0.980368,0.982717,1706,1.010950,1.010950,1.010950,1699,CKT_4892_11166
835,0.980839,0.978022,0.979799,1045,1.005887,1.005887,1.005887,978,CKT_1909_00720
836,0.942852,0.933449,0.944933,759,1.004700,1.004700,1.004700,625,CKT_3458_11675


In [15]:
import os, ctypes

# Pre-load KLUSolve.dll from py_dss_interface to avoid picking up CYME's copy
try:
    import py_dss_interface as _pdi
    _pdi_dll_dir = os.path.join(
        os.path.dirname(_pdi.__file__),
        'opendss_official', 'windows', 'delphi', 'x64',
    )
    _klu_path = os.path.join(_pdi_dll_dir, 'KLUSolve.dll')
    if os.path.isfile(_klu_path):
        if hasattr(os, 'add_dll_directory'):
            os.add_dll_directory(_pdi_dll_dir)
        ctypes.WinDLL(_klu_path)
except Exception:
    pass

from opendss.pf.powerflow import run_pf as dss_run_pf


def run_dss(dss_file):
    """Run OpenDSS power flow and return the result dict (keeps DSS session alive)."""
    result = dss_run_pf(dss_file)
    if not result['converged']:
        print(f"  WARNING: DSS did not converge for {dss_file}")
    else:
        print("DSS power flow converged")
    return result


def _get_dss_v_pu(dss_result):
    """Extract per-bus VA, VB, VC (pu) from an OpenDSS result dict."""
    d = dss_result['dss']
    rows = []
    for i in range(d.circuit.num_buses):
        d.circuit.set_active_bus_i(i)
        name = d.bus.name
        nodes = list(d.bus.nodes)          # e.g. [1, 2, 3]
        va_pu = d.bus.vmag_angle_pu        # [mag1, ang1, mag2, ang2, ...]

        v = {'NODE_ID': name, 'VA_DSS': None, 'VB_DSS': None, 'VC_DSS': None}
        phase_col = {1: 'VA_DSS', 2: 'VB_DSS', 3: 'VC_DSS'}
        for j, node in enumerate(nodes):
            if node in phase_col and j * 2 < len(va_pu):
                v[phase_col[node]] = va_pu[j * 2]
        rows.append(v)
    return pd.DataFrame(rows)


def compare_mc_dss(net, dss_file):
    """Run MC and OpenDSS, compute network-wide mean v_pu independently."""
    from multiconductor.pycci.cci_powerflow import run_pf

    # OpenDSS
    dss_result = run_dss(dss_file)
    df_dss = _get_dss_v_pu(dss_result)
    for c in ['VA_DSS', 'VB_DSS', 'VC_DSS']:
        df_dss[c] = pd.to_numeric(df_dss[c], errors='coerce')
    df_dss = df_dss.dropna(subset=['VA_DSS', 'VB_DSS', 'VC_DSS'])

    # MC
    run_pf(net, tol_vang_rad=1e-5, tol_vmag_pu=1e-5)
    df_mc = _get_mc_v_pu(net)
    for c in ['VA_MC', 'VB_MC', 'VC_MC']:
        df_mc[c] = pd.to_numeric(df_mc[c], errors='coerce')
    df_mc = df_mc.dropna(subset=['VA_MC', 'VB_MC', 'VC_MC'])

    summary = {
        'VA_MC_mean':   df_mc['VA_MC'].mean(),
        'VB_MC_mean':   df_mc['VB_MC'].mean(),
        'VC_MC_mean':   df_mc['VC_MC'].mean(),
        'MC_node_count': len(df_mc),
        'VA_DSS_mean':  df_dss['VA_DSS'].mean(),
        'VB_DSS_mean':  df_dss['VB_DSS'].mean(),
        'VC_DSS_mean':  df_dss['VC_DSS'].mean(),
        'DSS_node_count': len(df_dss),
    }
    return summary

In [16]:
def run_circuit2(circuit_key):
    import pickle
    from pathlib import Path

    print(f"Processing {circuit_key}...")
    pkl_file = Path(rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\{circuit_key}.pkl")
    dss_file = Path(rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\dss\{circuit_key}.dss")

    with open(pkl_file, 'rb') as fh:
        net = pickle.load(fh)

    summary = compare_mc_dss(net, str(dss_file))

    summary['CKT_KEY'] = circuit_key
    print(f"  {circuit_key}: MC nodes={summary['MC_node_count']} DSS nodes={summary['DSS_node_count']}, "
          f"VA mean MC={summary['VA_MC_mean']:.5f} DSS={summary['VA_DSS_mean']:.5f}")
    return summary

In [17]:
all_summaries2 = []
for circuit_key in df['CIRCUIT_KEY']:
    try:
        summary = run_circuit2(circuit_key)
        all_summaries2.append(summary)
    except Exception as e:
        print(f"Error running circuit {circuit_key}: {e}")

summary2_df = pd.DataFrame(all_summaries2)
summary2_df

Processing CKT_2155_12331...
  CKT_2155_12331: MC nodes=719 DSS nodes=1065, VA mean MC=0.94993 DSS=0.85242
Processing CKT_3250_02485...
  CKT_3250_02485: MC nodes=1399 DSS nodes=2219, VA mean MC=1.01167 DSS=0.81925
Processing CKT_156_06131...
  CKT_156_06131: MC nodes=479 DSS nodes=497, VA mean MC=0.98987 DSS=0.96438
Processing CKT_209_14375...
  CKT_209_14375: MC nodes=829 DSS nodes=898, VA mean MC=0.96746 DSS=0.93520
Processing CKT_1727_14043...
  CKT_1727_14043: MC nodes=763 DSS nodes=944, VA mean MC=1.09005 DSS=0.93266
Processing CKT_1747_14145...
DSS power flow converged
  CKT_1747_14145: MC nodes=2 DSS nodes=217, VA mean MC=1.00000 DSS=0.00922
Processing CKT_4372_05219...
  CKT_4372_05219: MC nodes=510 DSS nodes=591, VA mean MC=1.00104 DSS=0.89907
Processing CKT_1051_12960...
  CKT_1051_12960: MC nodes=1260 DSS nodes=1536, VA mean MC=0.99906 DSS=0.96227
Processing CKT_3953_09972...
DSS power flow converged
  CKT_3953_09972: MC nodes=407 DSS nodes=433, VA mean MC=0.99277 DSS=0.968

,VA_MC_mean,VB_MC_mean,VC_MC_mean,MC_node_count,VA_DSS_mean,VB_DSS_mean,VC_DSS_mean,DSS_node_count,CKT_KEY
0,0.949925,0.949001,0.990828,719,0.852418,0.855191,0.852251,1065,CKT_2155_12331
1,1.011674,1.090606,1.010080,1399,0.819246,0.818204,0.825515,2219,CKT_3250_02485
2,0.989875,0.977278,0.989813,479,0.964375,0.942441,0.964331,497,CKT_156_06131
3,0.967460,0.965021,0.972518,829,0.935202,0.935149,0.930923,898,CKT_209_14375
4,1.090048,1.072761,1.292763,763,0.932661,0.865583,0.838813,944,CKT_1727_14043
...,...,...,...,...,...,...,...,...,...
877,0.999293,0.999293,0.999292,391,0.989198,0.989197,0.989197,395,CKT_2082_11931
878,0.980956,0.980368,0.982717,1706,0.967987,0.964084,0.965268,1746,CKT_4892_11166
879,0.980839,0.978022,0.979799,1045,0.937624,0.927718,0.924618,1209,CKT_1909_00720
880,0.942852,0.933449,0.944933,759,0.852389,0.850458,0.872438,1067,CKT_3458_11675
